# RadioSkySims Uses

Fluxo de exemplo dos serviços do RadioSkySim. Todos os serviços usam uma
**data de referência comum** (1 mês antes de agora) e uma **janela de
24 h com passo de 15 min**. Recursos (fluxo solar, RINEX, TAP) são
buscados localmente (cache) e, se necessário, **online**; se nada for
encontrado, o notebook levanta uma **exceção clara** e segue para as
próximas células.


## 1. Parâmetros de referência (todos os serviços)

Aqui definimos a data de referência (``REF_DATE`` = 1 mês antes de
agora), a janela de simulação (24 h, passo 15 min) e o observador
padrão (BINGO). Todos os serviços abaixo usam estes parâmetros.


In [ ]:
# ── Âncora de diretório: raiz do projeto (CWD-independente) ──
# Garante que caminhos relativos (ex.: Path("data")) resolvam da raiz do
# projeto, independente de onde o kernel Jupyter iniciou (notebooks/ ou raiz).
import os

from radioskysims_core.paths import get_project_root

os.chdir(get_project_root())
print("cwd →", os.getcwd())

In [ ]:
# ── Parâmetros de referência compartilhados por TODOS os serviços ──
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np

# Data de referência: 1 mês antes de agora (UTC).
REF_DATE = (datetime.now(UTC) - timedelta(days=30)).replace(microsecond=0)
print("Data de referência (REF_DATE):", REF_DATE.isoformat())

# Janela de simulação: 24 h a partir de REF_DATE, passo de 15 min.
# TimeRange.from_iso aceita ISO naive (UTC) — não usa sufixo de timezone.
from radioskysims_core.domain.common import FrequencyRange, TimeRange

_ISO_FMT = "%Y-%m-%dT%H:%M:%S"
TR = TimeRange.from_iso(
    REF_DATE.strftime(_ISO_FMT),
    (REF_DATE + timedelta(hours=24)).strftime(_ISO_FMT),
    60 * 15,
)
print(
    "TimeRange (24 h, 15 min):",
    TR.to_datetime64().shape[0],
    "instantes | resolução:",
    TR.resolution_sec,
    "s",
)

FR = FrequencyRange(min_mhz=1000.0, max_mhz=1250.0, resolution_mhz=0.5)

# Observador de referência (BINGO — região do Uirapuru/RN).
from radioskysims_core.domain.common import Observer

OBSERVER = Observer(lat=-6.2391, lon=-38.9296, height=440.0, timeframe=TR)

# Diretórios de cache local para downloads.
DATA_DIR = Path("data")
RINEX_DIR = DATA_DIR / "rinex"
SOLAR_DIR = DATA_DIR / "solar_flux"
for _d in (RINEX_DIR, SOLAR_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Acesso a recursos: local (cache) → online → raise (e continua) ──
def load_or_download(name, local=None, download=None):
    """Resolve um recurso: usa o cache local se existir; senão baixa online.

    - ``local``: callable que retorna o recurso ou ``None``/``[]`` se ausente.
    - ``download``: callable que obtém o recurso online.
    - Se nem local nem online resolverem, levanta ``RuntimeError`` claro —
      o restante do notebook continua a partir das próximas células.
    """
    if local is not None:
        try:
            found = local()
            if found:
                print(f"[cache] {name}")
                return found
        except Exception:
            pass
    if download is None:
        raise RuntimeError(f"Recurso '{name}' indisponível: sem arquivo local e sem download.")
    print(f"[online] baixando {name} ...")
    try:
        return download()
    except Exception as exc:
        raise RuntimeError(
            f"Recurso '{name}' indisponível: local vazio e online falhou "
            f"({type(exc).__name__}: {exc})"
        ) from exc


## 2. Radiotelescópio & Beam Properties


In [ ]:
from radioskysims import create_radiotelescope

uirapuru = create_radiotelescope("default")

# Melhor __repr__ do RadiotelescopeSpec (agregado)
print(uirapuru)

# Antennas: __repr__() lista as propriedades disponíveis
print(uirapuru.antennas)

# Visão interativa (holoviews) do observador — mapa com tiles OSM
uirapuru.observer.view()


In [ ]:
# FWHM × frequência (gráfico holoviews)
uirapuru.antennas.get_properties("FWHM").view()
uirapuru.antennas.get_properties("Gain").view()
uirapuru.antennas.get_properties("BeamPattern").view()


## 3. Satellites

Usa o `SatelliteManager` com `REF_DATE` (a data de referência) e a janela
`TR` de 24 h. Requer PostgreSQL (`settings.toml`); se o catálogo local
estiver indisponível, o erro é exibido e o notebook continua.


In [ ]:
# Satélites via SatelliteOrchestrator (fachada) — TLE na data de referência
from radioskysims_sources import SatelliteOrchestrator

try:
    sat_orch = SatelliteOrchestrator.create_from_constellation(
        ["GPS", "GALILEO"],
        epoch=REF_DATE,
    )
    sats = sat_orch.satellites
    print(f"{len(sats)} satélites (deduplicados por NORAD ID) em {REF_DATE:%Y-%m-%d}")
    sat_orch.get_tle()[0]
except Exception as exc:
    print("[aviso] catálogo de satélites indisponível —", type(exc).__name__, str(exc)[:200])


In [ ]:
# build: monta o SatelliteAssembled (tracks + signals) na janela de 24 h
from radioskysims_viz import save_figure, save_plot

try:
    assembled = sat_orch.build(TR, FR)
    assembled  # __repr__ amigável (N satélites + dims)

    # Órbitas-terra (Mollweide): tracks por constelação + contornos de densidade
    fig = assembled.plot_tracks(by="constellation", contours=True)
    fig
    save_figure(fig, "sats_tracks_by_constellation")

    # Espectros PSD × frequência — uma curva por sinal (L1/L2/L5)
    ov = assembled.plot_signals()
    ov
    save_plot(ov, "sats_signals")

    # Seleção por constelação + sinais da seleção
    gps = assembled.select_Constellation("GPS")
    print(gps.summary())
    um = assembled.select_Satellite(noradID=gps.norad_ids[0])
    um.plot_signals()
except Exception as exc:
    print(
        "[aviso] montagem/plot de satélites indisponível (Postgres?) —",
        type(exc).__name__,
        str(exc)[:200],
    )


### Satélites a partir de RINEX (NAV broadcast)

Além do TLE (catálogo), os satélites podem ser criados a partir de
efemérides de transmissão (RINEX **NAV**) — download real (RBMC `pbcg`)
com cache em `data/rinex/`. O `RinexSatelliteManager` segue o **mesmo
fluxo do TLE**: criação → `build_tracks(TR)` → views, usando a mesma
janela de referência `TR`.

In [ ]:
# Satélites a partir de RINEX — download NAV/OBS + build(TR, FR) (mesmo fluxo do TLE)
from gnsstools4astro.application.services import RinexSatelliteManager
from radioskysims_viz import save_figure

STATION = "pbcg"  # estação RBMC próxima ao BINGO

try:
    # 1) manager configurado para a estação/data
    rmgr = RinexSatelliteManager.for_station(STATION, ref_date=REF_DATE, cache_dir=RINEX_DIR)
    rmgr

    # 2) download NAV (cache local ou online) — necessário p/ tracks/satélites
    nav_path = rmgr.download_nav()
    print("NAV:", nav_path)

    # 3) download OBS (cache local ou online) — usado pelo link budget
    obs_path = rmgr.download_obs()
    if obs_path:
        print("OBS:", obs_path)
    else:
        print("[info] OBS indisponível para a data — link budget não avaliado")

    # 4) build(TR, FR) — como o TLE: monta tracks + signals (PSD analítica)
    rinex_assembled = rmgr.build(TR, FR)
    rinex_assembled  # __repr__: N satélites + dims

    rinex_assembled.summary()
    rmgr.list_satellites()  # satélites carregados (como o get_tle() do TLE)
    rmgr.list_ephemerides()

    # 5) tracks na MESMA janela TR usada pelo TLE
    rinex_assembled.tracks
    fig = rinex_assembled.plot_tracks()
    fig
    save_figure(fig, "rinex_satellites_mollweide")

    # 6) signals — PSD ANALÍTICA (mesmo código do TLE): picos nas portadoras
    rinex_assembled.signals
    fig = rinex_assembled.plot_signals()
    fig
    save_figure(fig, "rinex_satellites_signals")

    # 7) link budget: C/N0 (obs) + range (nav) → EIRP estimada → signals.json
    if obs_path:
        import json as _json
        from gnsstools4astro.application.use_cases import EstimateLinkBudgetUseCase
        from gnsstools4astro.domain.constellations import get_constellation_signals

        estimates = rmgr.estimate_link_budget(obs_path)
        lb = EstimateLinkBudgetUseCase(get_constellation_signals())
        print("Link budget (obs RINEX) — EIRP estimada por banda:")
        for s in lb.summarize(estimates):
            print(
                f"  {s.constellation} {s.band_name:>5} @ {s.frequency_mhz:8.2f} MHz: "
                f"EIRP ~ {s.eirp_dbw_mean:6.2f} dBW (n={s.n_satellites})"
            )
        print("Fragmento signals.json (EIRP estimada):")
        print(
            _json.dumps(
                lb.to_signals_reference(estimates, base=get_constellation_signals()), indent=2
            )
        )
except Exception as exc:
    print("[aviso] RINEX indisponível (sem local/online) —", type(exc).__name__, str(exc)[:200])


## 4. Local Sources (Sun)

O Sol usa **fluxo solar real** (download SRD/SRS com cache local em
`data/solar_flux/`) e track via Skyfield no observador de referência.


In [ ]:
# Sun — fluxo solar REAL (SRD/SRS) com cache local + track via Skyfield
import pandas as pd
import xarray as xr

from radioskysims_viz import save_figure
from radioskysims_sources import LocalSourcesOrchestrator
from radioskysims_sources.domain.localsources import SolarFlux
from radioskysims_sources.infrastructure.external.solar_flux_downloader import (
    SolarFluxDownloader,
)

SOLAR_NC = SOLAR_DIR / f"solar_{REF_DATE:%Y%m%d}.nc"


def _solar_from_cache(nc: Path):
    ds = xr.open_dataset(nc)
    t = ds["time"].values.astype("datetime64[ns]")
    return SolarFlux(
        start_utc=pd.Timestamp(np.datetime64(t[0], "s")).to_pydatetime(),
        end_utc=pd.Timestamp(np.datetime64(t[-1], "s")).to_pydatetime(),
        dataset=ds,
        source_url=f"cache:{nc.name}",
    )


# Fluxo solar real (cache → online); se falhar, usa Sol calmo (fallback).
try:
    solar_flux = load_or_download(
        name="fluxo solar (SRD/SRS)",
        local=lambda: _solar_from_cache(SOLAR_NC) if SOLAR_NC.exists() else None,
        download=lambda: SolarFluxDownloader(timeout_sec=60).fetch(
            center_utc=REF_DATE, window_hours=24
        ),
    )
    if not SOLAR_NC.exists():
        try:
            solar_flux.dataset.to_netcdf(SOLAR_NC)
            print("  cache salvo:", SOLAR_NC)
        except Exception as _exc:
            print("  [aviso] não foi possível salvar cache do fluxo solar:", _exc)
    print("fluxo solar:", solar_flux.source_url)
except Exception as exc:
    print(
        "[aviso] fluxo solar online indisponível — usando Sol calmo:",
        type(exc).__name__,
        str(exc)[:200],
    )
    solar_flux = None

# Sun com o fluxo real (ou Sol calmo) + track no observador de referência
sun = LocalSourcesOrchestrator.create_sun(observer=OBSERVER, time_range=TR, flux=solar_flux)
print(sun.summary())
sun

# Carta do céu local ALT/AZ (matplotlib, deferido ao viz)
fig = sun.view_skychart()
fig
save_figure(fig, "sun_skychart")

# RA/Dec do Sol no mapa Mollweide (com continentes)
fig = sun.view_mollweide()
fig
save_figure(fig, "sun_mollweide")

# Fluxo solar × frequência (holder com .view())
fig = sun.flux_density().view()
fig
save_figure(fig, "sun_flux_density")

## 5. Radio Sources

O `RadioSourcesOrchestrator` carrega o **catálogo real** (banco `radio_sources`,
re-ingerido via TAP NVSS+SPECFIND) ou, sem banco, cai para um catálogo
simulado. Também é possível ingerir **NVSS + SPECFIND reais via TAP/ADQL
(VizieR)** na hora (rede necessária).

**Índice espectral (α):** fontes sem medida SPECFIND usam um **prior de
sincrotrão** (determinístico por fonte) — o catálogo real agora tem α
variando, em vez de todas com 0.7.

In [ ]:
# RadioSources: catálogo real (banco radio_sources) — re-ingerido via TAP
from radioskysims_viz import save_figure
from radioskysims_sources import RadioSourcesOrchestrator

# Carrega do banco (323 fontes NVSS reais, α variando). Alternativa demo:
#   radiosources = RadioSourcesOrchestrator.simulated(n_sources=40, seed=7)
radiosources = RadioSourcesOrchestrator.from_db()
radiosources  # __repr__ amigável (N fontes)

df = radiosources.list_properties()
print(
    "índices espectrais (α):",
    df["alpha"].nunique(),
    "únicos |",
    round(df["alpha"].min(), 3),
    "-",
    round(df["alpha"].max(), 3),
)
df

fig = radiosources.view_mollweide()
fig
save_figure(fig, "radio_sources_mollweide")

fig = radiosources.view_flux_density()
fig
save_figure(fig, "radio_sources_flux_density")

# Seleção por critérios + fluxo da seleção
bright = radiosources.select(min_flux_jy=1.0)
print(f"{len(bright)} fontes com S0 >= 1 Jy")
fig = bright.view_flux_density()
fig
save_figure(fig, "radio_sources_bright_flux")

# ── TAP real (VizieR): re-ingerir NVSS + SPECFIND numa janela do céu ──
# Fontes sem medida SPECFIND usam um prior de α (não ficam todas em 0.7).
try:
    tap = RadioSourcesOrchestrator.from_tap(
        ra_min_deg=180.0,
        ra_max_deg=200.0,
        dec_min_deg=-5.0,
        dec_max_deg=5.0,
        smin_jy=0.1,
    )
    print(f"{len(tap)} fontes NVSS reais (TAP VizieR)")
    tap_df = tap.list_properties()
    print("  α reais:", tap_df["alpha"].round(3).nunique(), "únicos")
    tap_df
    fig = tap.view_flux_density()
    fig
    save_figure(fig, "radio_sources_tap_flux")
except Exception as exc:
    print("[aviso] TAP indisponível (sem rede?) —", type(exc).__name__, str(exc)[:200])

## Maps